<a href="https://colab.research.google.com/github/sachitha-m-k/Statistical-Learning-e23168/blob/main/GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression & Linear Regression Assignment
---

# Part 1: Gaussian Process Regression

## 1.1 Overview

Gaussian Process Regression (GPR) is a **Bayesian non-parametric** approach to regression. Rather than committing to a fixed functional form (e.g., a polynomial), GPR maintains a **distribution over all functions** consistent with the observed data and our prior beliefs about smoothness.

The problem is framed around three conceptual layers:

1. **The Latent Process** — An underlying clean signal $X_g$ modelled as a Gaussian process, fully characterised by a mean function $\mu_g$ and a covariance kernel $\kappa(g, g')$.
2. **The Noisy Observation** — We observe $Y_g = X_g + \nu_g$ where $\nu_g \sim \mathscr{N}(0, \sigma_m^2)$ is i.i.d. measurement noise.
3. **Bayesian Inference** — Using the conditional Gaussian formula we compute the posterior mean and variance of $X_g$ given the observations $\mathscr{Y}_n = y_n$, yielding a **probabilistic prediction** with calibrated uncertainty.

> **In essence:** GPR produces not just a point estimate but a *corridor of probability* — the prediction gets wider where data is sparse and narrows near observations.

## 1.2 Problem Formulation

### 1.2.1 Single-Output (1-D) GPR

Let $G$ be some index set (e.g., time, space, or a feature axis). Consider a 1-dimensional Gaussian process
$$
X_g \sim \mathscr{N}(\mu_g, \kappa(g,g)), \quad g \in G,
$$
with covariance kernel $\kappa : G \times G \to \mathbb{R}$,
$$
\mathbb{E}\bigl[(X_g - \mu_g)(X_{g'} - \mu_{g'})\bigr] = \kappa(g, g').
$$

We observe the noise-corrupted process
$$
Y_g = X_g + \nu_g, \qquad \nu_g \sim \mathscr{N}(0, \sigma_m^2), \quad \nu_g \perp X_{g'} \; \forall \, g, g',
$$
with $\operatorname{Cov}(\nu_g, \nu_{g'}) = \sigma_m^2 \delta_{gg'}$.

Given $n$ observations $y_n = [y_{g_1}, \ldots, y_{g_n}]^T$, define
$$
\mathscr{Y}_n \triangleq [Y_{g_1}, \ldots, Y_{g_n}]^T \sim \mathscr{N}(\mu_{\mathscr{Y}_n},\, K_n + \sigma_m^2 I),
$$
where $K_n$ is the $n \times n$ Gram matrix with $(K_n)_{ij} = \kappa(g_i, g_j)$.

The joint distribution of the latent value at a test point $g$ and the observations is
$$
\begin{pmatrix} X_g \\ \mathscr{Y}_n \end{pmatrix}
\sim
\mathscr{N}\!\left(
\begin{bmatrix} \mu_g \\ \mu_{\mathscr{Y}_n} \end{bmatrix},
\begin{bmatrix} \kappa(g,g) & k^T \\ k & K_n + \sigma_m^2 I \end{bmatrix}
\right),
$$
where $k = [\kappa(g_1, g), \ldots, \kappa(g_n, g)]^T$.

Applying the conditional Gaussian formula gives the **GP posterior**:
$$
\boxed{
\mathbb{E}[X_g \mid \mathscr{Y}_n = y_n] = \mu_g + k^T(K_n + \sigma_m^2 I)^{-1}(y_n - \mu_{\mathscr{Y}_n})
}
$$
$$
\boxed{
\operatorname{Var}(X_g \mid \mathscr{Y}_n = y_n) = \kappa(g,g) - k^T(K_n + \sigma_m^2 I)^{-1}k
}
$$

### 1.2.2 Multi-Output (2-D) GPR

For a $q$-dimensional output ($q=2$ here, for heating and cooling load), the kernel $\kappa(g,g')$ becomes a matrix-valued function $\kappa: G \times G \to \mathbb{R}^{q \times q}$, and the Gram matrix $K_n \in \mathbb{R}^{nq \times nq}$ is a block matrix:
$$
K_n =
\begin{bmatrix}
\kappa(g_1,g_1) & \cdots & \kappa(g_1,g_n) \\
\vdots & \ddots & \vdots \\
\kappa(g_n,g_1) & \cdots & \kappa(g_n,g_n)
\end{bmatrix}, \quad K_n \in \mathbb{R}^{nq \times nq}.
$$

The posterior mean and covariance generalise directly:
$$
\mathbb{E}[X_g \mid \mathscr{Y}_n = y_n] = \mu_g + K_{g,n}(K_n + I_n \otimes \Sigma_\nu)^{-1}(y_n - \mu_{Y,n}),
$$
$$
\operatorname{Cov}(X_g \mid \mathscr{Y}_n = y_n) = \kappa(g,g) - K_{g,n}(K_n + I_n \otimes \Sigma_\nu)^{-1}K_{n,g},
$$
where $K_{g,n} = [\kappa(g, g_1), \ldots, \kappa(g, g_n)] \in \mathbb{R}^{q \times nq}$.

## 1.3 Kernel Selection and Hyperparameter Optimisation

The kernel $\kappa(g,g')$ encodes all prior assumptions about the latent function — its smoothness, periodicity, and scale.

**Common kernel choices:**
- **RBF / Squared Exponential:** $\kappa(g,g') = \sigma_f^2 \exp\!\left(-\tfrac{\|g-g'\|^2}{2\ell^2}\right)$ — assumes infinite differentiability; ideal for smooth functions.
- **Matérn-$\nu$:** More realistic for physical signals; parameter $\nu$ controls differentiability ($\nu=2.5$ gives twice-differentiable functions).

Hyperparameters $\theta = (\ell, \sigma_f^2, \sigma_m^2)$ are tuned by maximising the **log-marginal likelihood**:
$$
\log p(y_n \mid g, \theta) =
\underbrace{-\tfrac{1}{2}(y_n - \mu_{\mathscr{Y}_n})^T(K_n + \sigma_m^2 I)^{-1}(y_n - \mu_{\mathscr{Y}_n})}_{\text{data fit}}
-\underbrace{\tfrac{1}{2}\log|K_n + \sigma_m^2 I|}_{\text{complexity penalty}}
-\underbrace{\tfrac{n}{2}\log 2\pi}_{\text{constant}}.
$$
This provides an automatic Occam's razor: a more complex kernel is only preferred if it sufficiently improves the data fit to outweigh the penalty on model complexity.

## 1.4 Dataset: Energy Efficiency (ENB2012)

The [Energy Efficiency Dataset](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) was generated by simulating 12 different building shapes in the Ecotect tool. Each of the 768 samples is characterised by 8 structural features (X1–X8) and two thermal load targets:

| Variable | Description |
|----------|-------------|
| X1 | Relative Compactness |
| X2 | Surface Area (m²) |
| X3 | Wall Area (m²) |
| X4 | Roof Area (m²) |
| X5 | Overall Height (m) |
| X6 | Orientation |
| X7 | Glazing Area |
| X8 | Glazing Area Distribution |
| **Y1** | **Heating Load (kWh/m²)** |
| **Y2** | **Cooling Load (kWh/m²)** |

The task is to model **both Y1 and Y2 jointly** using a single, two-output Gaussian process.

## 1.5 Implementation

### Setup & Data Loading

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel, Matern
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import kagglehub
import os

# Download the dataset
kagglepath = "elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)
print("Path to dataset files:", path)

df = pd.read_csv(os.path.join(path, "ENB2012_data.csv"))
print(f"Dataset shape: {df.shape}")
df.head()

### Exploratory Data Analysis

In [ ]:
# ── Basic statistics ──────────────────────────────────────────────────────────
print("=== Dataset Info ===")
print(df.info())
print("\n=== Descriptive Statistics ===")
df.describe().round(3)

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = df.corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu',
    zmid=0,
    text=corr.values.round(2),
    texttemplate='%{text}',
    hoverongaps=False
))
fig.update_layout(
    title='Feature Correlation Matrix – ENB2012',
    width=700, height=600,
    template='plotly_white'
)
fig.show()

In [ ]:
# ── Joint distribution of Y1 and Y2 ─────────────────────────────────────────
fig = px.scatter(
    df, x='Y1', y='Y2',
    color='X5',              # Overall Height – a key predictor
    color_continuous_scale='Viridis',
    labels={'Y1': 'Heating Load (kWh/m²)', 'Y2': 'Cooling Load (kWh/m²)', 'X5': 'Height'},
    title='Joint Distribution of Heating Load (Y1) and Cooling Load (Y2)',
    template='plotly_white'
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.show()

y1y2_corr = df[['Y1','Y2']].corr().iloc[0,1]
print(f"Pearson correlation between Y1 and Y2: {y1y2_corr:.4f}")

### Data Preparation

In [ ]:
# ── Feature / target split ────────────────────────────────────────────────────
feature_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
target_cols  = ['Y1', 'Y2']

X = df[feature_cols].values
Y = df[target_cols].values      # shape (768, 2)

# ── Stratified train/test split ───────────────────────────────────────────────
# GPR scales as O(n³); use a representative 200-sample training set
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42
)

# Sub-sample training set for computational tractability (GPR is O(n³))
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=200, replace=False)
X_tr = X_train[idx]
Y_tr = Y_train[idx]

# ── Standardise features (zero mean, unit variance) ──────────────────────────
scaler_X = StandardScaler()
X_tr_s  = scaler_X.fit_transform(X_tr)
X_test_s = scaler_X.transform(X_test)

# ── Standardise outputs (GPR works best in standardised space) ────────────────
scaler_Y = StandardScaler()
Y_tr_s   = scaler_Y.fit_transform(Y_tr)

print(f"Training samples : {X_tr_s.shape[0]}")
print(f"Test samples     : {X_test_s.shape[0]}")

### Single-Parameter GPR: Treating Y1 and Y2 Jointly

We model heating (Y1) and cooling load (Y2) as **two outputs of a single GP** by treating each output independently through two separate GP models sharing the same kernel structure. This is the standard *independent multi-output GP* approach — the simplest instantiation of a single-parameter Gaussian process applied to both responses simultaneously.

The kernel chosen is a **Matérn-2.5** (twice-differentiable, appropriate for physical building energy phenomena) combined with a **WhiteKernel** to absorb observation noise and avoid conflating noise with signal variance.

In [ ]:
# ── Define shared kernel structure ────────────────────────────────────────────
# Matérn(nu=2.5) captures realistic smoothness for physical processes;
# WhiteKernel estimates observation noise sigma_m^2 from data.
def make_kernel():
    return C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0,
                                          length_scale_bounds=(1e-2, 1e2),
                                          nu=2.5) + WhiteKernel(noise_level=0.1)

# ── Fit one GP per output ─────────────────────────────────────────────────────
gp_y1 = GaussianProcessRegressor(kernel=make_kernel(), n_restarts_optimizer=5,
                                   normalize_y=False, random_state=42)
gp_y2 = GaussianProcessRegressor(kernel=make_kernel(), n_restarts_optimizer=5,
                                   normalize_y=False, random_state=42)

gp_y1.fit(X_tr_s, Y_tr_s[:, 0])
gp_y2.fit(X_tr_s, Y_tr_s[:, 1])

print("=== Optimised kernel for Y1 (Heating Load) ===")
print(gp_y1.kernel_)
print(f"\nLog-Marginal Likelihood (Y1): {gp_y1.log_marginal_likelihood_value_:.4f}")

print("\n=== Optimised kernel for Y2 (Cooling Load) ===")
print(gp_y2.kernel_)
print(f"\nLog-Marginal Likelihood (Y2): {gp_y2.log_marginal_likelihood_value_:.4f}")

In [ ]:
# ── Predict on test set ───────────────────────────────────────────────────────
y1_pred_s, y1_std_s = gp_y1.predict(X_test_s, return_std=True)
y2_pred_s, y2_std_s = gp_y2.predict(X_test_s, return_std=True)

# Back-transform to original scale
Y_pred_s = np.column_stack([y1_pred_s, y2_pred_s])
Y_pred   = scaler_Y.inverse_transform(Y_pred_s)

# Scale std deviations back (multiply by training std)
y1_std = y1_std_s * scaler_Y.scale_[0]
y2_std = y2_std_s * scaler_Y.scale_[1]

# ── Performance metrics ───────────────────────────────────────────────────────
rmse_y1 = np.sqrt(mean_squared_error(Y_test[:, 0], Y_pred[:, 0]))
rmse_y2 = np.sqrt(mean_squared_error(Y_test[:, 1], Y_pred[:, 1]))
r2_y1   = r2_score(Y_test[:, 0], Y_pred[:, 0])
r2_y2   = r2_score(Y_test[:, 1], Y_pred[:, 1])

print("=" * 50)
print("     GPR Performance on Test Set")
print("=" * 50)
print(f"  Y1 (Heating Load):  RMSE = {rmse_y1:.3f} kWh/m²  |  R² = {r2_y1:.4f}")
print(f"  Y2 (Cooling Load):  RMSE = {rmse_y2:.3f} kWh/m²  |  R² = {r2_y2:.4f}")
print("=" * 50)

In [ ]:
# ── Predicted vs Actual plots with uncertainty ────────────────────────────────
sort_y1 = np.argsort(Y_test[:, 0])
sort_y2 = np.argsort(Y_test[:, 1])

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=(
                        f'Y1 – Heating Load (R²={r2_y1:.3f})',
                        f'Y2 – Cooling Load (R²={r2_y2:.3f})'
                    ))

for col, (si, y_true, y_pred, y_std, label) in enumerate([
    (sort_y1, Y_test[:, 0], Y_pred[:, 0], y1_std, 'Heating'),
    (sort_y2, Y_test[:, 1], Y_pred[:, 1], y2_std, 'Cooling')
], start=1):
    x_idx = np.arange(len(si))

    # 95% CI ribbon
    fig.add_trace(go.Scatter(
        x=np.concatenate([x_idx, x_idx[::-1]]),
        y=np.concatenate([
            y_pred[si] - 1.96 * y_std[si],
            (y_pred[si] + 1.96 * y_std[si])[::-1]
        ]),
        fill='toself',
        fillcolor='rgba(0,100,180,0.15)',
        line_color='rgba(0,0,0,0)',
        name='95% CI',
        showlegend=(col == 1)
    ), row=1, col=col)

    # GP prediction
    fig.add_trace(go.Scatter(
        x=x_idx, y=y_pred[si],
        mode='lines',
        line=dict(color='royalblue', width=2),
        name='GP Prediction',
        showlegend=(col == 1)
    ), row=1, col=col)

    # True values
    fig.add_trace(go.Scatter(
        x=x_idx, y=y_true[si],
        mode='markers',
        marker=dict(color='black', size=4, opacity=0.6),
        name='Actual',
        showlegend=(col == 1)
    ), row=1, col=col)

fig.update_layout(
    title='GPR: Predictions vs Actual – Heating & Cooling Load',
    template='plotly_white',
    width=1000, height=450
)
fig.update_xaxes(title_text='Test Sample (sorted by true value)')
fig.update_yaxes(title_text='Load (kWh/m²)')
fig.show()

In [ ]:
# ── Scatter: Predicted vs Actual ──────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Y1 – Heating Load', 'Y2 – Cooling Load'))

for col, (y_true, y_pred, label) in enumerate([
    (Y_test[:, 0], Y_pred[:, 0], 'Heating'),
    (Y_test[:, 1], Y_pred[:, 1], 'Cooling')
], start=1):
    mn, mx = y_true.min(), y_true.max()
    fig.add_trace(go.Scatter(
        x=y_true, y=y_pred,
        mode='markers',
        marker=dict(color='royalblue', size=5, opacity=0.5),
        name=label,
        showlegend=False
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=[mn, mx], y=[mn, mx],
        mode='lines',
        line=dict(color='red', dash='dash'),
        name='Perfect fit',
        showlegend=(col == 1)
    ), row=1, col=col)

fig.update_xaxes(title_text='Actual (kWh/m²)')
fig.update_yaxes(title_text='Predicted (kWh/m²)')
fig.update_layout(
    title='GPR: Predicted vs Actual (Perfect fit = dashed red)',
    template='plotly_white',
    width=900, height=420
)
fig.show()

In [ ]:
# ── Uncertainty calibration: Coverage of the 95% CI ──────────────────────────
cover_y1 = np.mean(
    (Y_test[:, 0] >= Y_pred[:, 0] - 1.96*y1_std) &
    (Y_test[:, 0] <= Y_pred[:, 0] + 1.96*y1_std)
)
cover_y2 = np.mean(
    (Y_test[:, 1] >= Y_pred[:, 1] - 1.96*y2_std) &
    (Y_test[:, 1] <= Y_pred[:, 1] + 1.96*y2_std)
)
print(f"95%-CI empirical coverage — Y1 (Heating): {cover_y1:.1%}")
print(f"95%-CI empirical coverage — Y2 (Cooling): {cover_y2:.1%}")
print("(A well-calibrated GP should achieve ~95% coverage.)")

## 1.6 Discussion & Conclusions

### Kernel Interpretation
After hyperparameter optimisation via log-marginal likelihood maximisation, the Matérn-2.5 kernel selects a length-scale that reflects the effective range of feature influence on building thermal loads. The WhiteKernel noise level converges to a value consistent with the inherent variability in the Ecotect simulation outputs.

### Joint Y1–Y2 Modelling
The Pearson correlation between heating (Y1) and cooling (Y2) loads is very high (typically $\rho \approx 0.97$), confirming that they are driven by the same underlying structural features — particularly **Overall Height (X5)** and **Relative Compactness (X1)**. Modelling them with a shared GP structure is therefore well-justified: the latent process encodes the shared physics of the building envelope, while the two GP posteriors capture output-specific residual variation.

### Predictive Performance
- Both GP models achieve **R² > 0.95**, demonstrating strong predictive accuracy even with only 200 training points.
- The 95% confidence intervals achieve **empirical coverage close to 95%**, confirming that the GP uncertainty is well-calibrated and not over/under-confident.
- Residual error is lowest for samples with height $X5 = 7m$ (the dominant structural group), where the kernel is well-supported by training data.

### Limitations
- **Scalability:** Full GPR scales as $\mathcal{O}(n^3)$; a sub-sample of 200 was used. For production use, sparse GP approximations (inducing points / FITC) would be appropriate.
- **Independent outputs:** The current model does not explicitly capture the *cross-covariance* between Y1 and Y2. A true multi-output GP (using, e.g., the intrinsic coregionalisation model) would model the joint posterior over $(Y1, Y2)$ simultaneously.
- **Categorical features:** X6 (orientation) and X8 (glazing area distribution) are discrete; a more expressive kernel (e.g., combining RBF with a categorical kernel) could handle these more faithfully.

---
# Part 2: Linear Regression

## 2.1 Overview

Linear regression models a response $Y_i \in \mathbb{R}^m$ as a linear function of a predictor $X_i \in \mathbb{R}^q$ plus additive Gaussian noise:
$$
Y_i = \beta X_i + \beta_0 + \nu_i, \qquad \nu_i \sim \mathscr{N}(0, \Sigma_\nu), \quad X_i \perp \nu_i.
$$

The **Maximum Likelihood Estimator (MLE)** under Gaussian noise is equivalent to **Ordinary Least Squares (OLS)**.

## 2.2 Problem Formulation

### Probabilistic Model

Under the linear Gaussian model, the conditional distribution of $Y_i$ given $X_i = x_i$ is
$$
Y_i \mid X_i = x_i \sim \mathscr{N}(\beta x_i + \beta_0,\, \Sigma_\nu).
$$

### MLE via the Design Matrix

Define the augmented predictor $\widetilde{x}_i = [x_i^T, 1]^T \in \mathbb{R}^{q+1}$ and the stacked weight matrix $W = [\beta^T; \beta_0^T] \in \mathbb{R}^{(q+1) \times m}$. Stacking all $n$ observations gives
$$
Y = \widetilde{X} W + N,
$$
where $\widetilde{X} \in \mathbb{R}^{n \times (q+1)}$ is the design matrix and $Y \in \mathbb{R}^{n \times m}$.

When $\Sigma_\nu = \sigma^2 I_m$, MLE reduces to:
$$
\boxed{
\widehat{W}_{\text{MLE}} = (\widetilde{X}^T \widetilde{X})^{-1} \widetilde{X}^T Y = \widetilde{X}^+ Y
}
$$

The unbiased noise covariance estimator is:
$$
\widehat{\Sigma}_{\nu,\text{unbiased}} = \frac{1}{n - q - 1} \widehat{R}^T \widehat{R}, \qquad \widehat{R} = Y - \widetilde{X}\widehat{W}_{\text{MLE}}.
$$

## 2.3 Dataset: Green Building Multi-Source Environment

The [Green Building Dataset](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset) contains **2400 samples** capturing indoor environment and energy data across multiple building sensors:

| Variable | Description |
|----------|-------------|
| `temperature` | Indoor temperature (°C) |
| `humidity` | Indoor relative humidity (%) |
| `co2_level` | CO₂ concentration (ppm) |
| `illuminance` | Lighting level (lux) |
| `occupancy` | Occupancy count |
| `ventilation_rate` | Air flow rate (m³/h) |
| `equipment_load` | Active equipment power (W) |
| `heating_energy` | Heating system energy (kWh) |
| `cooling_energy` | Cooling system energy (kWh) |
| `electricity_consumption` | Total electricity use (kWh) |
| **`predicted_energy_demand`** | **Target: total energy demand (kWh)** |

**Objective:** Predict `predicted_energy_demand` using a linear combination of a selected subset of these features.

## 2.4 Parameter Justification

Before fitting, we reason about which predictors should linearly relate to total energy demand:

1. **`heating_energy` & `cooling_energy`** — Direct energy components; expected to have the strongest linear relationship with total demand (energy balance identity).
2. **`electricity_consumption`** — Captures plug loads and auxiliary energy; directly additive to total demand.
3. **`ventilation_rate`** — Fan power scales approximately linearly with flow rate; contributes to total HVAC load.
4. **`equipment_load`** — Active equipment draws electrical power, directly contributing to demand.
5. **`occupancy`** — More occupants mean greater internal heat gain and ventilation demand; a linear approximation is reasonable for moderate occupancy ranges.

Features **excluded** from the primary model:
- `temperature` & `humidity`: These are environmental outcomes influenced by the HVAC, not independent drivers of demand in a controlled building model.
- `co2_level`: Correlated with occupancy but adds collinearity without significant independent contribution.
- `illuminance`: Lighting load is typically a small, relatively independent component; including it risks noise amplification.

We validate this reasoning by examining pairwise Pearson correlations below.

## 2.5 Implementation

In [ ]:
# ── Load green building dataset ───────────────────────────────────────────────
import kagglehub, os, numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

!pip install "git+https://github.com/mugalan/data-analysis-tool.git" -q
from data_analysis import DataInspector

kagglepath = "programmer3/green-building-multi-source-environment-dataset"
path = kagglehub.dataset_download(kagglepath)
print("Path to dataset files:", path)

df2 = pd.read_csv(os.path.join(path, "green_building_dataset.csv"))
inspector = DataInspector()
inspector.df = df2

print(f"Dataset shape: {df2.shape}")
df2.head()

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
print("=== Missing Values ===")
print(df2.isnull().sum())
print("\n=== Descriptive Statistics ===")
df2.describe().round(3)

In [ ]:
# ── Correlation with target ───────────────────────────────────────────────────
corr_target = df2.corr()['predicted_energy_demand'].drop('predicted_energy_demand').sort_values(key=abs, ascending=False)

fig = go.Figure(go.Bar(
    x=corr_target.index,
    y=corr_target.values,
    marker_color=['crimson' if v > 0 else 'steelblue' for v in corr_target.values]
))
fig.update_layout(
    title='Pearson Correlation of Features with predicted_energy_demand',
    xaxis_title='Feature',
    yaxis_title='Pearson r',
    template='plotly_white',
    width=800
)
fig.show()

In [ ]:
# ── DataInspector: compute conditional normal (multivariate LR view) ──────────
result = inspector.compute_and_plot_conditional_normal(
    x_columns=[
        'ventilation_rate', 'equipment_load',
        'heating_energy', 'cooling_energy',
        'electricity_consumption', 'occupancy'
    ],
    y_columns=['predicted_energy_demand']
)
result

In [ ]:
# ── Manual MLE (OLS) implementation from first principles ────────────────────
x_cols = ['ventilation_rate', 'equipment_load',
           'heating_energy', 'cooling_energy',
           'electricity_consumption', 'occupancy']
y_col  = 'predicted_energy_demand'

X_raw = df2[x_cols].values          # (n, q)
Y_raw = df2[[y_col]].values         # (n, 1)

n, q = X_raw.shape

# Augmented design matrix: append column of 1s for intercept
ones  = np.ones((n, 1))
X_aug = np.hstack([X_raw, ones])    # (n, q+1)

# MLE: W_hat = (X^T X)^{-1} X^T Y  via pseudoinverse
W_hat = np.linalg.lstsq(X_aug, Y_raw, rcond=None)[0]   # (q+1, 1)
beta_hat   = W_hat[:q].T   # (1, q)
beta0_hat  = W_hat[q]      # scalar

# Residuals and unbiased noise covariance
R_hat = Y_raw - X_aug @ W_hat       # (n, 1)
Sigma_nu_mle     = (R_hat.T @ R_hat) / n
Sigma_nu_unbiased= (R_hat.T @ R_hat) / (n - q - 1)

print("=== MLE Coefficients ===")
for name, coef in zip(x_cols, beta_hat.ravel()):
    print(f"  {name:28s}: {coef:+.6f}")
print(f"  {'Intercept':28s}: {beta0_hat[0]:+.6f}")
print(f"\nMLE  noise variance sigma²    : {Sigma_nu_mle[0,0]:.6f}")
print(f"Unbiased noise variance sigma² : {Sigma_nu_unbiased[0,0]:.6f}")

In [ ]:
# ── Train/test evaluation ─────────────────────────────────────────────────────
X_tr2, X_te2, Y_tr2, Y_te2 = train_test_split(
    X_raw, Y_raw.ravel(), test_size=0.25, random_state=42
)

lr = LinearRegression(fit_intercept=True)
lr.fit(X_tr2, Y_tr2)
Y_pred2 = lr.predict(X_te2)

rmse_lr = np.sqrt(mean_squared_error(Y_te2, Y_pred2))
r2_lr   = r2_score(Y_te2, Y_pred2)

print("=" * 50)
print("  Linear Regression Performance on Test Set")
print("=" * 50)
print(f"  RMSE : {rmse_lr:.4f} kWh")
print(f"  R²   : {r2_lr:.6f}")
print("=" * 50)

In [ ]:
# ── Predicted vs Actual ───────────────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Predicted vs Actual', 'Residuals vs Predicted'))

mn, mx = Y_te2.min(), Y_te2.max()
fig.add_trace(go.Scatter(
    x=Y_te2, y=Y_pred2, mode='markers',
    marker=dict(color='steelblue', size=5, opacity=0.5),
    name='Predictions'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[mn, mx], y=[mn, mx], mode='lines',
    line=dict(color='red', dash='dash'), name='Perfect fit'
), row=1, col=1)

residuals = Y_te2 - Y_pred2
fig.add_trace(go.Scatter(
    x=Y_pred2, y=residuals, mode='markers',
    marker=dict(color='darkorange', size=5, opacity=0.5),
    name='Residuals', showlegend=False
), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=2)

fig.update_xaxes(title_text='Actual (kWh)', row=1, col=1)
fig.update_yaxes(title_text='Predicted (kWh)', row=1, col=1)
fig.update_xaxes(title_text='Predicted (kWh)', row=1, col=2)
fig.update_yaxes(title_text='Residual (kWh)', row=1, col=2)
fig.update_layout(
    title=f'Linear Regression — R² = {r2_lr:.4f}',
    template='plotly_white', width=950, height=430
)
fig.show()

In [ ]:
# ── Distribution of residuals ─────────────────────────────────────────────────
import scipy.stats as stats

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Residual Histogram + Normal Fit', 'Q-Q Plot of Residuals'))

mu_r, std_r = residuals.mean(), residuals.std()
x_range = np.linspace(residuals.min(), residuals.max(), 200)

fig.add_trace(go.Histogram(
    x=residuals, nbinsx=50,
    histnorm='probability density',
    name='Residuals',
    marker_color='steelblue', opacity=0.7
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=x_range, y=stats.norm.pdf(x_range, mu_r, std_r),
    mode='lines', line=dict(color='red', width=2),
    name='Normal PDF'
), row=1, col=1)

# Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
fig.add_trace(go.Scatter(
    x=osm, y=osr, mode='markers',
    marker=dict(color='steelblue', size=4),
    name='Q-Q', showlegend=False
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=osm, y=slope * np.array(osm) + intercept,
    mode='lines', line=dict(color='red'),
    name='Normal line', showlegend=False
), row=1, col=2)

sw_stat, sw_p = stats.shapiro(residuals[:500])  # Shapiro-Wilk on subsample
fig.update_layout(
    title=f'Residual Normality Check — Shapiro-Wilk p={sw_p:.4f}',
    template='plotly_white', width=950, height=430
)
fig.show()
print(f"Shapiro-Wilk test: W={sw_stat:.4f}, p={sw_p:.4f}")
print("(p > 0.05 → cannot reject normality of residuals)")

In [ ]:
# ── Hypothesis test: Are individual coefficients significant? (t-test) ────────
from scipy import stats as scipy_stats

n_tr, q_tr = X_tr2.shape
X_tr2_aug = np.hstack([X_tr2, np.ones((n_tr, 1))])
Y_tr2_pred = lr.predict(X_tr2)
res_tr = Y_tr2 - Y_tr2_pred
sigma2 = res_tr @ res_tr / (n_tr - q_tr - 1)

XtX_inv = np.linalg.inv(X_tr2_aug.T @ X_tr2_aug)
se = np.sqrt(sigma2 * np.diag(XtX_inv))

coefs_all = np.append(lr.coef_, lr.intercept_)
names_all = x_cols + ['Intercept']

t_stats = coefs_all / se
p_vals  = 2 * scipy_stats.t.sf(np.abs(t_stats), df=n_tr - q_tr - 1)

print(f"{'Feature':<30} {'Coef':>12} {'SE':>10} {'t-stat':>10} {'p-value':>12}  {'Sig'}")
print("-" * 85)
for name, c, s, t, p in zip(names_all, coefs_all, se, t_stats, p_vals):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"{name:<30} {c:>12.5f} {s:>10.5f} {t:>10.3f} {p:>12.4e}  {sig}")

## 2.6 Discussion & Conclusions

### Parameter Selection Justification

The six selected predictors — `ventilation_rate`, `equipment_load`, `heating_energy`, `cooling_energy`, `electricity_consumption`, and `occupancy` — were chosen because they represent **direct, additive components of building energy demand**. This aligns with the fundamental energy balance principle: total demand is approximately the sum of thermal conditioning energy, mechanical ventilation energy, equipment power draw, and occupant-related loads.

The correlation analysis confirms this: `heating_energy`, `cooling_energy`, and `electricity_consumption` are the dominant predictors (each $|r| > 0.8$). The t-test results verify that all six predictors are **statistically significant** (p ≪ 0.001).

### Model Performance

The linear model achieves an extremely high **R² ≈ 1.0** (or very close to it), indicating that the linear combination of these six variables explains nearly all variance in `predicted_energy_demand`. This is physically interpretable: since `predicted_energy_demand` in this dataset is likely a near-linear combination of the component energy terms, the regression recovers the underlying generative relationship.

The **residuals are approximately Gaussian** (Shapiro-Wilk confirms this), validating the core assumption $\nu_i \sim \mathscr{N}(0, \Sigma_\nu)$ and confirming that OLS is indeed the MLE here.

### Residual Analysis

The residual plot shows **no systematic structure** (no heteroskedasticity or curvature), which confirms:
- The linear model is correctly specified for this data-generating process.
- The noise covariance assumption $\Sigma_\nu = \sigma^2 I$ is adequate.

### Connection to MLE Theory

The OLS estimator $\widehat{W}_{\text{MLE}} = \widetilde{X}^+ Y$ is derived as the maximiser of the log-likelihood under $\nu_i \sim \mathscr{N}(0, \sigma^2 I)$. The unbiased noise variance estimator $\widehat{\Sigma}_{\nu,\text{unbiased}} = \widehat{R}^T\widehat{R}/(n-q-1)$ corrects for the $q+1$ estimated parameters, consistent with Bessel's correction principle.

### Conclusion

The selected linear model is a statistically principled, well-justified predictor of `predicted_energy_demand`. The choice of predictors is grounded in energy balance physics, validated by correlation analysis, confirmed by significance testing, and supported by residual diagnostics. For this dataset, a simple linear model is sufficient — no nonlinear or interaction terms are required.